# W13-D5 概念实验：行业能力包 = 契约打包——用代码复刻 Capability Catalog 的注册规则

配套阅读：`第13周-Day5-VisionCapabilityArchitecture-行业能力包如何炼成.md`（md 讲架构，本 notebook 用**可执行实验**验证三个核心概念）：

1. **契约不可变**：复刻 `lnkchat/capability/catalog.py` 的 frozen 描述符 + immutable 字段 + 版本升级规则；
2. **正交 facet**：Capability × Industry 二维矩阵，行业词禁入 Capability ID（ADR-003 硬约束的机器可校验形式）；
3. **打包 = 元数据组装**：Application = capabilities 子集 × industries 标签（ADR-004 application.yaml 的组装器），并量化「同一组能力卖两个行业」的复用率。

只用 numpy / matplotlib / 标准库。所有规则均从真实代码/ADR 提炼，实验里的注册器是教学级最小复刻。


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


## 实验一：最小 CapabilityRegistry——把 ADR-003 与真实 catalog.py 的规则写成可校验的代码

对照 `catalog.py` 的四条硬规则：ID 正则、版本正则、frozen+immutable（同 id@version 改契约必须拒绝）、lifecycle。
实验新增两条**治理前移**规则（来自今天 md 第5节的分析）：行业词禁入 ID（ADR-003 §2.2.1）、destructive 能力必须挂人审（镜像 skill_release 的 conditional_write → human_review 交叉校验）。


In [ ]:
import re
from dataclasses import dataclass, field

ID_RE = re.compile(r"^[a-z0-9]+(\.[a-z0-9]+)*$")
VER_RE = re.compile(r"^v[0-9]+$")
# ADR-003：行业词只允许出现在 Application 的 industries 标签，禁止进入 Capability ID 段
INDUSTRY_WORDS = {"retail", "mall", "finance", "manufacturing", "bank", "hospital", "logistics"}
# catalog.py _IMMUTABLE_FIELDS 的教学版
IMMUTABLE = {"input_schema", "output_schema", "effects", "required_scopes", "approval_policy", "provider", "execution_mode"}

@dataclass(frozen=True)
class CapabilityDescriptor:
    capability_id: str
    capability_version: str
    lifecycle: str = "published"          # draft / published / deprecated
    effects: str = "read"                 # read / write / destructive
    approval_policy: str = "none"         # none / runtime_human_approval
    provider: str = "lnkchat"
    input_schema: dict = field(default_factory=dict)
    output_schema: dict = field(default_factory=dict)

    def __post_init__(self):
        if not ID_RE.match(self.capability_id):
            raise ValueError(f"ID 不合法(须小写点分): {self.capability_id!r}")
        if not VER_RE.match(self.capability_version):
            raise ValueError(f"版本不合法(须 vN): {self.capability_version!r}")
        bad = set(self.capability_id.split(".")) & INDUSTRY_WORDS
        if bad:
            raise ValueError(f"行业词 {bad} 禁入 Capability ID（ADR-003 正交约束）: {self.capability_id!r}")
        if self.effects == "destructive" and self.approval_policy != "runtime_human_approval":
            raise ValueError(f"destructive 能力必须挂人审（治理前移交叉校验）: {self.capability_id!r}")
        if self.lifecycle not in {"draft", "published", "deprecated"}:
            raise ValueError(f"lifecycle 非法: {self.lifecycle!r}")

class CapabilityRegistry:
    """复刻 catalog.py 的 CapabilityRegistry：register 冲突检测 + deprecate。"""
    def __init__(self):
        self._entries: dict[tuple[str, str], CapabilityDescriptor] = {}
    def register(self, d: CapabilityDescriptor):
        key = (d.capability_id, d.capability_version)
        if key in self._entries:
            old = self._entries[key]
            fields = IMMUTABLE & set(old.__dict__)  # 教学版描述符字段子集
            changed = {f for f in fields if getattr(old, f) != getattr(d, f)}
            if changed:
                raise ValueError(
                    f"契约不可变：{key[0]}@{key[1]} 重注册时 {sorted(changed)} 发生变化 → 必须升版本")
        self._entries[key] = d
    def deprecate(self, cid: str, ver: str):
        old = self._entries[(cid, ver)]
        self._entries[(cid, ver)] = CapabilityDescriptor(
            **{**old.__dict__, "lifecycle": "deprecated"})
    def published(self):
        return [d for d in self._entries.values() if d.lifecycle == "published"]

reg = CapabilityRegistry()
# 镜像真实 P0（catalog.py _register_p0_capabilities）
for d in [
    CapabilityDescriptor("lnkchat.knowledge.query", "v1"),
    CapabilityDescriptor("lnkchat.workflow.execute", "v1", approval_policy="runtime_human_approval"),
]:
    reg.register(d)
# 今天的 vision 候选（md 思考题①的三能力方案 + effects 裁决）
for d in [
    CapabilityDescriptor("lnkchat.vision.detect", "v1", effects="write"),   # 触发抓拍+GPU：算成本按 write 记
    CapabilityDescriptor("lnkchat.vision.kpi.query", "v1"),
    CapabilityDescriptor("lnkchat.safety.alert.summary", "v1"),
]:
    reg.register(d)
print(f"已注册 {len(reg.published())} 个 published 能力:")
for d in reg.published():
    print(f"  {d.capability_id}@{d.capability_version}  effects={d.effects:<11} approval={d.approval_policy}")


## 实验二：四个「被拒绝」的注册——约束的价值在于它挡住了什么

逐条触发：① 行业词入 ID；② 大小写/下划线不合规；③ 同版本偷改契约；④ destructive 不挂人审。最后用**升版本**修复③。


In [ ]:
def try_register(factory, label):
    """factory 延迟构造：拒绝可能发生在 __post_init__（构造期）或 register（注册期），都要接住"""
    try:
        reg.register(factory())
        return f"OK    {label}"
    except ValueError as e:
        return f"REJECT {label}\n       -> {e}"

print(try_register(lambda: CapabilityDescriptor("lnkchat.retail.vision.query", "v1"),
      "① 行业词 retail 进 ID（想做『零售专用』能力）"))
print(try_register(lambda: CapabilityDescriptor("lnkchat.Vision.detect", "v1"),
      "② 大写段（想用品牌风格命名）"))
print(try_register(lambda: CapabilityDescriptor("lnkchat.vision_detect", "v1"),
      "② 下划线（Python 习惯）"))
print(try_register(lambda: CapabilityDescriptor(
      "lnkchat.vision.detect", "v1", effects="write", output_schema={"boxes": "array"}),
      "③ 同版本偷改 output_schema（想悄悄加字段）"))
print(try_register(lambda: CapabilityDescriptor("lnkchat.camera.baseline.reset", "v1",
      effects="destructive", approval_policy="none"),
      "④ destructive 且无人审（想一键重置全部基线图）"))

# 正确姿势：制造业要安全帽字段 → 发 v2，v1 转 deprecated
reg.register(CapabilityDescriptor("lnkchat.vision.detect", "v2", effects="write",
    output_schema={"boxes": "array", "classes": "array", "helmet_flag": "bool?"}))
reg.deprecate("lnkchat.vision.detect", "v1")
print()
print("修复演示：lnkchat.vision.detect@v2 发布（output_schema 增补 helmet_flag），v1 转 deprecated")
print("契约演进的正确姿势 = 新版本并行 + 旧版本日落，而不是原地偷改")


## 实验三：Capability × Industry 正交矩阵（ADR-003 §2 的可视化）

行=Capability（行业无关），列=Industry（应用层标签）。空 cell 合法（不适用就留空）。**行业覆盖率和填充率**是正交设计的经济学读数。


In [ ]:
import numpy as np

caps = ["vision.detect", "vision.kpi.query", "safety.alert.summary",
        "knowledge.query", "workflow.execute"]
inds = ["retail", "manufacturing", "logistics", "finance"]
# 适用性矩阵（1=该行业客户会买这项能力）——来自业务判断，空 cell 合法
M = np.array([
    [1, 1, 1, 0],   # vision.detect：三大实体场景都需要，金融不需要
    [1, 1, 1, 0],   # kpi.query
    [1, 1, 1, 0],   # safety.alert.summary
    [1, 1, 1, 1],   # knowledge.query：全行业
    [1, 1, 1, 1],   # workflow.execute：全行业
])
fill = M.sum() / M.size
print(f"矩阵填充率 {fill:.0%}（ADR-003 §2.2.5：空 cell 不是缺陷）")
for c, r in zip(caps, M.sum(axis=1) / len(inds)):
    print(f"  lnkchat.{c:<22} 行业覆盖率 {r:.0%}")
print()
print("关键读数：通用能力（knowledge/workflow）覆盖 100%，视觉能力覆盖 75%。")
print("若按行业复制能力（lnkchat.retail.vision.* / lnkchat.mfg.vision.*），同样 3 项视觉功能要注册 9 个 ID；")
print("正交设计只注册 3 个——行业差异被压缩成一张标签表，这就是『能力包』的经济学。")

fig, ax = plt.subplots(figsize=(8, 4.2))
im = ax.imshow(M, cmap="YlGn", vmin=0, vmax=1.3, aspect="auto")
ax.set_xticks(range(len(inds)), inds)
ax.set_yticks(range(len(caps)), ["lnkchat." + c for c in caps])
for i in range(len(caps)):
    for j in range(len(inds)):
        ax.text(j, i, "V" if M[i, j] else "-", ha="center", va="center",
                fontsize=14, fontweight="bold", color="white" if M[i, j] else "gray")
ax.set_title("Capability × Industry 正交 facet 矩阵（ADR-003）\n行=行业无关能力 · 列=应用层标签 · 空 cell 合法")
fig.colorbar(im, ax=ax, shrink=0.8, label="适用")
plt.tight_layout()
plt.savefig("/root/learning-notebooks/第13周/w13d5_facet_matrix.png", dpi=130)
plt.show()


## 实验四：能力包组装器——Application = capabilities 子集 × industries 标签

复刻 ADR-004 application.yaml 的组装+校验：能力必须已注册且 published、行业标签必须来自冻结清单、名字必须符合 LnkChat AI *X* 模板（ADR-005）。然后用**同一组能力**打两个行业的包，验证「包=元数据」。


In [ ]:
import json

FROZEN_INDUSTRIES = {"retail", "manufacturing", "logistics", "finance"}  # ADR-003 industries.yaml
NAME_RE = re.compile(r"^LnkChat AI [A-Z][a-z]+$")                                # ADR-005 模板

def assemble_application(name, capabilities, industries, registry):
    """ADR-004 application.yaml 组装器：三重校验后产出元数据包。"""
    assert NAME_RE.match(name), f"应用名必须匹配 LnkChat AI *X*（ADR-005）: {name!r}"
    unknown = set(industries) - FROZEN_INDUSTRIES
    assert not unknown, f"行业标签未冻结: {unknown}（增删须走新 ADR）"
    have = {(d.capability_id, d.capability_version) for d in registry.published()}
    missing = [c for c in capabilities if tuple(c.split("@")) not in have]
    assert not missing, f"引用了未注册/已废弃能力: {missing}"
    return {"application": {"name": name, "capabilities": capabilities,
                           "industries": industries, "status": "ga"}}

# 错误演示：引用 deprecated 的 v1 → 组装失败
try:
    assemble_application("LnkChat AI Vision", ["lnkchat.vision.detect@v1"], ["retail"], reg)
except AssertionError as e:
    print("被拒绝:", e)

# 正确：引用 v2。同一组能力，打两个行业的 SKU
core = ["lnkchat.vision.detect@v2", "lnkchat.vision.kpi.query@v1",
        "lnkchat.safety.alert.summary@v1", "lnkchat.knowledge.query@v1",
        "lnkchat.workflow.execute@v1"]
sku_retail = assemble_application("LnkChat AI Vision", core, ["retail"], reg)
sku_plant  = assemble_application("LnkChat AI Vision", core, ["manufacturing", "logistics"], reg)
print(json.dumps(sku_retail, ensure_ascii=False, indent=2))
print()
print("复用率验证：2 个 SKU（零售版/工厂版）共用同一组 5 个 Capability 描述符；")
print("Vision Runtime（MallSenseAI backend/workers）代码 0 行改动——行业差异全部收敛在 industries 标签。")
print("这就是『能力包 = 契约打包』：打包动作只写元数据，不碰段3 的运行时。")


## 实验结论

1. **约束即架构**：四条注册拒绝规则（正则/行业词/immutable/destructive 须人审）全部可以机器校验——ADR 从「文档共识」变成「类型系统守门」。被挡住的每个错误注册，都是一次未来的事故。
2. **契约演进 = 加版本，不是偷改**：vision.detect v1→v2（加 helmet_flag）+ v1 deprecated 的并行日落模式，是双行业客户灰度的最小机制。
3. **正交矩阵的经济学**：3 项视觉功能 × 4 行业，行业复制式命名要 9 个 ID，正交设计只要 3 个 + 一张标签表；组装器证明**打包动作只产出元数据**（application.yaml），段3 的 Runtime 一行代码不动——「行业能力包」包的是契约，不是代码。
